<h1 style="text-align: center; font-family: 'menlo'; color: #9b1515; font-size:50px;">
  <span style="background-color: #e6a3a3; padding: 5px 10px; border-radius: 5px; display: inline-block;">
         Combine calibrated dark images for use in later reduction step
  </span>
</h1>

In [2]:
from pathlib import Path
from astropy.nddata import CCDData
from astropy.stats import mad_std

import ccdproc as ccdp
import matplotlib.pyplot as plt
import numpy as np

from convenience_functions import show_image
plt.style.use('guide.mplstyle')

# Recommended settings for image combination

As discussed in the [notebook about combining images](https://www.astropy.org/ccd-reduction-and-photometry-guide/v/dev/notebooks/01-06-Image-combination.html), <u>the recommendation is that you combine by averaging the individual images but sigma clip to remove extreme values.</u>

[ccdproc](https://ccdproc.readthedocs.org/) provides two ways to combine:

- An object-oriented interface built around the <span style="color: #d62495">Combiner</span> object, described in the [ccdproc documentation on image combination.](https://ccdproc.readthedocs.io/en/latest/image_combination.html)

- A function called [combine](https://ccdproc.readthedocs.io/en/latest/api/ccdproc.combine.html#ccdproc.combine), which we will use here because the function allows you to specify the maximum amount of memory that should be used during combination. That feature can be essential depending on how many images you need to combine, how big they are, and how much memory your computer has.

**NOTE: If using a version of ccdproc lower than 2.0, set the memory limit a factor of 2-3 lower than you want the maximum memory consumption to be.**

# Example 1: Cryogenically-cooled camera

The remainder of this section assumes that the calibrated bias images are in the folder <span style="color: #d62495">example1-reduced</span> which was created in the previous notebook.

In [4]:
calibrated_path = Path ('example1-reduced')
reduced_images = ccdp.ImageFileCollection(calibrated_path)

## Make a combined image for each exposure time in Example 1

There are several dark exposure times in this data set. By converting the times in the summary table to a set it returns only the unique values.

In [5]:
darks = reduced_images.summary['imagetyp'] == 'DARK'
dark_times = set(reduced_images.summary['exptime'][darks])
print(dark_times)

{np.float64(300.0), np.float64(70.0), np.float64(7.0)}


The code below <u>loops</u> over the dark exposure times and, for each exposure time:

- selects the relevant calibrated dark images,

- combines them using the <span style="color: #d62495">combine</span> function,

- adds the keyword <span style="color: #d62495">COMBINED</span> to the header so that later calibration steps can easily identify which bias to use, and

- writes the file whose name includes the exposure time.

In [ ]:
for exp_time in sorted(dark_times):
    calibrated_darks = reduced_images.files_filtered(imagetyp='dark',
                                                    exptime=exp_time,
                                                    include_path=True)
    combined_dark = ccdp.combine(calibrated_darks,
                                method='average',
                                sigma_clip=True, sigma_clip_low_thresh=5,
                                sigma_clip_high_thresh=5, sigma_clip_func=np.ma.median,
                                sigma_clip_dev_func=mad_std,mem_limit=350e6)
    combined_dark.meta['combined']=True
    dark_file_name = 'combined_dark{:6.3f}.fit'.format(exp_time)
    combined_dark.write(calibrated_path / dark_file_name)

INFO:astropy:splitting each image into 4 chunks to limit memory usage to 350000000.0 bytes.


INFO: splitting each image into 4 chunks to limit memory usage to 350000000.0 bytes. [ccdproc.combiner]


INFO:astropy:splitting each image into 4 chunks to limit memory usage to 350000000.0 bytes.


INFO: splitting each image into 4 chunks to limit memory usage to 350000000.0 bytes. [ccdproc.combiner]


<h1 style="text-align: left; font-family: 'menlo'; color: #e6a3a3; font-size:40px;">
  <span style="background-color: #9b1515; padding: 5px 10px; border-radius: 5px; display: inline-block;">
         Links
  </span>
</h1>

https://github.com/sh4rkNy/astropy-coisas/tree/main<br>https://github.com/astropy/ccd-reduction-and-photometry-guide<br>https://www.astropy.org/ccd-reduction-and-photometry-guide/v/dev/notebooks/03-06-Combine-darks-for-use-in-later-calibration-steps.html